# BRCA Spatial Transcriptomics — Notebook 1
## Data → QC → spatial graph → GNN → spatial domains

This is the first of three notebooks for the project: **spatial organisation of the
breast-tumour microenvironment** on a 10x Visium slide.

**What this notebook produces:** a set of *spatial domains* — regions of the slide that
group together because they are both molecularly similar **and** physically close. We get
them by training a small **graph neural network (GNN)** that we write ourselves, so the
message-passing mechanism is fully visible rather than hidden inside a package.

Pipeline in this notebook:
1. Download the Visium breast-cancer slide (counts + H&E image + spot coordinates)
2. Quality control (MAD-based, scverse style) and normalisation
3. Build the **spatial graph** (spots = nodes, adjacency = edges)
4. Train our **graph autoencoder** → a spatially-aware embedding per spot
5. Cluster the embeddings → **spatial domains**, and visualise them on the tissue
6. Save the result for Notebook 2 (deconvolution)

> Run on **Colab with a GPU runtime** (Runtime → Change runtime type → T4 GPU).
> The data download reaches 10x servers, so it needs a normal internet connection.

## 0 · Setup

Install everything, then mount Google Drive so downloaded data and the output `.h5ad`
persist between Colab sessions (sessions time out, and we don't want to re-download or
re-train every time).

`torch-geometric` is the library that gives us graph-neural-network layers. The wheels
match your installed PyTorch/CUDA automatically on recent versions.

In [ ]:
# --- install (quiet) ---
!pip install -q scanpy squidpy leidenalg torch-geometric

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import scanpy as sc
import squidpy as sq
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.utils import from_scipy_sparse_matrix
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("scanpy", sc.__version__, "| torch", torch.__version__, "| device:", DEVICE)

In [ ]:
# --- mount Drive for persistence (optional but recommended on Colab) ---
from google.colab import drive
drive.mount("/content/drive")

import os
WORKDIR = "/content/drive/MyDrive/brca_spatial"   # change if you like
os.makedirs(WORKDIR, exist_ok=True)
print("Saving everything to:", WORKDIR)

## 1 · Download the Visium slide

We use the public **10x Visium human breast cancer** slide (`Block A, Section 1`). scanpy
fetches the Space Ranger output: the spot × gene count matrix, the spot coordinates
(`adata.obsm["spatial"]`), and the H&E histology image (`adata.uns["spatial"]`, which we
keep for the morphology step in Notebook 3).

Each row of `adata` is a **spot** (~55 µm of tissue, roughly 1–10 cells), not a single cell.

In [ ]:
adata = sc.datasets.visium_sge(sample_id="V1_Breast_Cancer_Block_A_Section_1")
adata.var_names_make_unique()
print(adata)
print("\nspots:", adata.n_obs, "| genes:", adata.n_vars)

## 2 · Quality control

Same idea as single-cell QC, applied to spots. We compute three metrics per spot —
total counts, number of genes detected, and % mitochondrial reads — then flag **outliers**
using the **median absolute deviation (MAD)** rather than fixed cutoffs. MAD adapts to the
dataset and is the scverse-recommended approach: a spot is an outlier if it sits more than
`n_mads` MADs from the median.

We keep filtering **permissive** so we don't throw away real tissue regions.

In [ ]:
# annotate mitochondrial genes (human -> 'MT-' prefix) and compute QC metrics
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None,
                           log1p=False, inplace=True)

from scipy.stats import median_abs_deviation

def is_mad_outlier(adata, metric, n_mads=5):
    # True for spots that are >n_mads MADs from the median of `metric`
    x = adata.obs[metric]
    med = np.median(x)
    mad = median_abs_deviation(x)
    return (x < med - n_mads * mad) | (x > med + n_mads * mad)

# flag outliers on the count/gene metrics (log-scale is gentler for skewed counts)
adata.obs["log1p_total_counts"]      = np.log1p(adata.obs["total_counts"])
adata.obs["log1p_n_genes_by_counts"] = np.log1p(adata.obs["n_genes_by_counts"])

outlier = (
    is_mad_outlier(adata, "log1p_total_counts", 5)
    | is_mad_outlier(adata, "log1p_n_genes_by_counts", 5)
    | is_mad_outlier(adata, "pct_counts_mt", 3)      # stricter on MT%
    | (adata.obs["pct_counts_mt"] > 20)              # hard ceiling on MT%
)
print(f"flagged {int(outlier.sum())} / {adata.n_obs} spots as low quality")

In [ ]:
# quick look before filtering
fig, axs = plt.subplots(1, 3, figsize=(12, 3))
axs[0].hist(adata.obs["total_counts"], bins=60);       axs[0].set_title("total counts")
axs[1].hist(adata.obs["n_genes_by_counts"], bins=60);  axs[1].set_title("genes/spot")
axs[2].hist(adata.obs["pct_counts_mt"], bins=60);      axs[2].set_title("% MT")
plt.tight_layout(); plt.show()

In [ ]:
# apply spot filter, then drop genes seen in very few spots
adata = adata[~outlier].copy()
sc.pp.filter_genes(adata, min_cells=3)
print("after QC ->", adata.n_obs, "spots,", adata.n_vars, "genes")

## 3 · Normalisation & features

We **stash the raw counts** in a layer first — Notebook 2's deconvolution model (DestVI)
needs raw counts, so we must not lose them. Then the usual steps: normalise each spot to a
common total, `log1p`, pick highly variable genes, scale, and run PCA. The **50 PCs become
the input node features** for the GNN (denser and less noisy than the full gene matrix).

In [ ]:
adata.layers["counts"] = adata.X.copy()          # <-- keep raw counts for Notebook 2

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata                                   # log-normalised, all genes (for plotting later)

sc.pp.highly_variable_genes(adata, n_top_genes=2000)
adata = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(adata, max_value=10)
sc.pp.pca(adata, n_comps=50)
print("PCA node features:", adata.obsm["X_pca"].shape)

## 4 · Build the spatial graph

This is the step that makes the analysis *spatial*. Visium spots sit on a hexagonal grid,
so each interior spot has up to **6 immediate neighbours**. `squidpy` builds that
neighbour graph from the coordinates and stores it as a sparse adjacency matrix in
`adata.obsp["spatial_connectivities"]`.

We then convert that adjacency into the **`edge_index`** format PyTorch Geometric expects:
a `2 × E` tensor listing every connected pair `(source, target)`. The node features are the
50 PCs from the previous step.

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="grid", n_neighs=6)
A = adata.obsp["spatial_connectivities"]            # sparse adjacency (spots x spots)

edge_index, _ = from_scipy_sparse_matrix(A)         # -> 2 x E tensor of (src, dst)
x = torch.tensor(adata.obsm["X_pca"], dtype=torch.float)

edge_index = edge_index.to(DEVICE)
x = x.to(DEVICE)
print("nodes:", x.shape[0], "| node feature dim:", x.shape[1], "| edges:", edge_index.shape[1])

## 5 · The graph autoencoder (the GNN we build ourselves)

This is the heart of the notebook. A **graph autoencoder (GAE)** has two parts:

- **Encoder** — two `GCNConv` layers. A `GCNConv` *is* the message-passing step: for every
  spot it averages the feature vectors of that spot **and its graph neighbours**, then
  applies a learned linear transform + nonlinearity. One layer mixes information across 1
  hop; stacking two layers reaches 2 hops. The output is a low-dimensional **embedding**
  `z` per spot that encodes both its own expression *and* its spatial neighbourhood.
- **Decoder** — a plain linear layer that tries to reconstruct the original node features
  from `z`. Training to reconstruct forces `z` to be informative.

Because every `GCNConv` blends a spot with its neighbours, neighbouring spots end up with
similar embeddings — which is exactly what we want before clustering into spatial domains.

The model is ~15 lines. Read the `forward` path top to bottom: `x → conv1 → relu → conv2
→ z`, then `z → decoder → x_hat`. The loss is how far `x_hat` is from `x`.

In [ ]:
class SpatialGAE(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, latent_dim=16):
        super().__init__()
        self.conv1   = GCNConv(in_dim,  hidden_dim)   # message passing, hop 1
        self.conv2   = GCNConv(hidden_dim, latent_dim) # message passing, hop 2 -> embedding
        self.decoder = nn.Linear(latent_dim, in_dim)   # reconstruct node features from z

    def encode(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))          # aggregate neighbours, then ReLU
        z = self.conv2(h, edge_index)                  # aggregate again -> spatial embedding
        return z

    def forward(self, x, edge_index):
        z = self.encode(x, edge_index)
        x_hat = self.decoder(z)                         # try to rebuild the input
        return z, x_hat

model = SpatialGAE(in_dim=x.shape[1]).to(DEVICE)
print(model)

In [ ]:
# --- training loop (explicit, so you can watch the loss fall) ---
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-4)

model.train()
for epoch in range(1, 201):
    optimizer.zero_grad()
    z, x_hat = model(x, edge_index)
    loss = F.mse_loss(x_hat, x)         # reconstruction loss
    loss.backward()
    optimizer.step()
    if epoch % 25 == 0 or epoch == 1:
        print(f"epoch {epoch:3d}   recon loss {loss.item():.4f}")

# --- pull out the learned spatial embedding for every spot ---
model.eval()
with torch.no_grad():
    z, _ = model(x, edge_index)
adata.obsm["X_gnn"] = z.cpu().numpy()
print("\nspatial embedding stored in adata.obsm['X_gnn']:", adata.obsm["X_gnn"].shape)

## 6 · Cluster the embeddings → spatial domains

Now the same move as your BRCA project, but on the **spatially-aware** embedding instead of
raw expression: build a neighbour graph in embedding space and run Leiden clustering. Each
cluster is a **spatial domain**. Tune `resolution` up for more (finer) domains, down for
fewer.

In [ ]:
sc.pp.neighbors(adata, use_rep="X_gnn", n_neighbors=15)
sc.tl.leiden(adata, resolution=0.5, key_added="domain")
print(adata.obs["domain"].value_counts())

In [ ]:
# domains drawn on the tissue — this is the payoff plot
sq.pl.spatial_scatter(adata, color="domain", size=1.4, figsize=(7, 7),
                      title="GNN spatial domains")

In [ ]:
# UMAP of the embedding, coloured by domain (sanity check: domains should separate)
sc.tl.umap(adata)
sc.pl.umap(adata, color="domain", title="GNN embedding (UMAP)")

## 7 · Save for Notebook 2

We write the AnnData with the domain labels, the raw-count layer, and the spatial graph.
Notebook 2 picks this up, brings in a reference single-cell atlas, and estimates the
cell-type composition of every spot (deconvolution).

In [ ]:
out_path = os.path.join(WORKDIR, "brca_visium_domains.h5ad")
adata.write(out_path)
print("saved ->", out_path)

---
### Where we are
You now have data-driven spatial domains from a GNN you trained yourself. Things to try
before moving on: change `resolution` in the Leiden step to see domains split/merge, or
change `latent_dim` / number of `GCNConv` layers in the model and watch how the domains
shift. When you're happy, tell me and I'll build **Notebook 2 (deconvolution)**.